This notebook will run the code for analysis of BD Rhapsody human BCR files for detection of CLL Clones, which can be removed in Cellismo. 
Python version 3.12.9
Date started: 04-Jan-2026
Author: Amelia Fisher
Based on: https://scverse.org/scirpy/tags/v0.11.0/generated/scirpy.io.read_airr.html#scirpy.io.read_airr, https://scirpy.scverse.org/en/latest/tutorials/tutorial_5k_bcr.html
 
Run on: Aire High Performance Computer, University of Leeds, UK

The control data must first be analysed in BCR_controls.ipynb.

Input requirements:

1) VDJ_Dominant_Contigs_AIRR file from BD Rhapsody Valsera (formerly SevenBridges) primary analysis pipeline. Alternative methods to generate AIRR files and AIRR file definition can be found elsewhere and are beyond the scope of this notebook. 

2) Note that the AIRR file will contain all reads from a multiplexed batch if multiplexing by sample tag (SMK) has been used on the BD Rhapsody platform. If individual sample files are required, these will need to be generated individually (using a for loop to separate out tags per cell_id or similar) prior to ue of this pipeline.

3) No batch correction is required for the dataset this was designed for. This is because the dominant BCR contig is either present or absent, it does not have a relative expression value like standard RNA seq. Check to see if this assumption applies to your data too. QC is run here in preprocessing. 

4) H5mu file of transcriptomic data which can be edited and generated in BDRhapsody cellismo before exporting as this file format and being used here. For clone identification it is not batch corrected because only the metadata e.g. sample names and study arms, is used. For transcriptomic analysis in conjucntion with BCR see BCR.ipynb, where transcriptomics will require batch correction. 



## Set-up ##

In [ ]:
# Import python modules

import numpy as np
import pandas as pandas
import scanpy as sc
import scirpy as ir 
import muon as mu
import skbio as sk
import os as os
from muon import MuData
from matplotlib import pyplot as plt, cm as mpl_cm
from cycler import cycler

#sc.set_figure_params(figsize=(6, 6))  ## generates an error, get this code for current versions if its needed.
sc.settings.verbosity = 2  # verbosity: errors (0), warnings (1), info (2), hints (3)

In [ ]:
sc.logging.print_header()

# Set-up paths etc

In [ ]:
import os

# change paths throughout script

# Batch 1 data
#multi_sample = False
#work_folder = "PATH/runFolders/Batch1_test"
#h5mu_file = "Batch1_usedForTests_muon_v2"
#airr_file = "Batch1_VDJ_Dominant_Contigs_AIRR.tsv"

# Not that batch5 had a processing error at the library prep stage, which resulted in very low quantity of the BCR amplifying. This had lead to a lot of unassigned BCR CDR3 nt sequences in this dataset.

# Batch 1 + 2 - increase depending on how many batches are used.
multi_sample = True
work_folder = "PATH/runFolders/Batch1_Batch2_test"
h5mu_file = "Batch1and2.h5mu"
airr_file = "Batch1_VDJ_Dominant_Contigs_AIRR.tsv"
airr_file_2 = "Batch2_VDJ_Dominant_Contigs_AIRR.tsv"

## Get the data in ##

Get the data in, tutorial available at: https://scverse.org/scirpy/tags/v0.11.0/tutorials/tutorial_io.html#importing-data

Use BOTH transcriptomic and BCR data in this analysis

In [ ]:
# Get the transcriptomic data in, which has the group categories. 
# If I need different groupings I can make these in cellismo, and export as scanpy h5ad, and read as this file type instead. 
# AnnData and MuData options given here. 

#adata = ir.io.read_h5ad(PATH/Data/Testfiles/ipynb_test_data/Batch1_Test")

#adata.shape

#### MAKE SURE THE CORRECT FILE IS BEING CALLED HERE ####

#mdata = ir.io.read_h5mu("PATH/Data/Batch1and2/Batch1and2.h5mu")

mdata = ir.io.read_h5mu(os.path.join(work_folder, h5mu_file))

mdata.shape

In [ ]:
# Check my mdata object columns

mdata.obs.columns

#mdata.obs.head()

In [ ]:
# Check data layout

mdata.obs.head()

In [ ]:
# Multi-sample if there is different notatation between columns
mdata.obs.rename(columns={"Sample_Name": "Sample_Tag_Name"})

# Check my data object columns
mdata.obs.columns

# Filter out Multiplets and Undetermined first

This should be run for Cellismo processed results to remove any multiplet and undetermined categories that have left an imprint (i.e. empty columns etc) on the dataset, even if they have had the multiplets and undteremined cells removed in Cellismo. 

Doublet detection is not run because scrublet by scanpy detects doublets on a single sample by differences in cell type transcriptome. This does not apply to this dataset. Use the automatically detected multiple removal (based on sample tags) instead.



In [ ]:
# Filter out Sample Tags without reliable data. 

mask = ~mdata.obs["Sample_Name"].isin(["Multiplet", 
                                      "Undetermined", 
                                       "<unassigned>"])
mdata_filtered = mdata[mask, :].copy()

# Step through all columns and remove any unused categories
import pandas as pd

for col in mdata_filtered.obs.columns:
    if pd.api.types.is_categorical_dtype(mdata_filtered.obs[col].dtype):
        mdata_filtered.obs[col] = mdata_filtered.obs[col].cat.remove_unused_categories()

mdata = mdata_filtered
mdata.shape

In [ ]:
# Filter out BCRs that are unassigned to a nt sequence OR filter out specific nucleotide BCR sequences. # change BCR CDR3 nt sequences
# Also removes unassigned data, because these can become apparently dominant clones based on other parts of the VDJ, but I cannot process them accurately for analysis.
# Removal of unassigned BCRs = slightly less data, but much more accurate data.  

mask = ~mdata.obs["BCR_Heavy_CDR3_Junction_Nucleotide_Dominant"].isin(["<unassigned>",
"TGTGCGAGAGAAAGGGACCAACTGGCAGAGCACTGG",
"TGTGCGAAACATCTTCGATATAGCGGCTCGTCAGGAGATGCTTTTGATATCTGG",
"TGTGCGAAAGATATGTCCCTCTGGTGGCAGTGGCTTATTGACTACTACTACTACTACATGGACGTCTGG",
"TGTGCGCGATGGAGTAAATATTACTGGTACTTCGATCTCTGG",
"TGTGCGCGATGGAGTAAATATTACTGGTACTTCGATCTCTGG",
"TGTGTGAGAGATGCTATCTGCTATACGAAATGTGTCCACTACTACTACATGGACGTCTGG",
"TGTGCGAGAGGGACTAGTGGGAGCTACGTCGACTACTGG",]
)
mdata_filtered = mdata[mask, :].copy()

# Step through all columns and remove any unused categories
import pandas as pd

for col in mdata_filtered.obs.columns:
    if pd.api.types.is_categorical_dtype(mdata_filtered.obs[col].dtype):
        mdata_filtered.obs[col] = mdata_filtered.obs[col].cat.remove_unused_categories()

mdata = mdata_filtered
mdata.shape

In [ ]:
# Check my mdata object columns

mdata.obs.head()

In [ ]:
# Get the AIRR file in with Anndata
# AnnData objects to merge AIRR with MuData downstream.
# The most recent BD Rhapsody primary anaylsis pipeline on Velsera (Seven Bridges) puts it into a standard AIRR format, so read_airr is used NOT read_bd_rhapsody!
# Defaults used for read_airr as per documentation https://scverse.org/scirpy/tags/v0.11.0/generated/scirpy.io.read_airr.html
# Dominant chains used in this analysis, path to dominant chain AIRR file. Unfiltered chains not needed at this stage.
# If I am importing more than one AnnData object, call each something different e.g. adata_1, adata_2 etc.
# If I need different groupings I can make these in cellismo, and export as scanpy h5ad, and read as this file type instead.

# adata = scirpy.io.read_airr("PATH/Data/Batch1/Batch1_VDJ_Dominant_Contigs_AIRR.tsv", use_umi_count_col='auto', infer_locus=True, cell_attributes=('is_cell', 'high_confidence', 'multi_chain'), include_fields=('productive', 'locus', 'v_call', 'd_call', 'j_call', 'c_call', 'junction', 'junction_aa', 'consensus_count', 'duplicate_count'))

# adata_ir = ir.io.read_airr("PATH/Data/Batch1/Batch1_VDJ_Dominant_Contigs_AIRR.tsv")
# adata_ir = ir.io.read_airr(os.path.join(work_folder, airr_file))


##################################################################
# -----------------PRE-PROCESS AIRR FILES-------------------------#
# -----------------CHANGE BATCH NUMBERS---------------------------#
##################################################################
if multi_sample:
    print("Processing multiple samples AIRR files before merging into MuData object.")
    import pandas

    batch1_airr = pandas.read_csv(os.path.join(work_folder, airr_file), sep="\t")
    batch1_airr["cell_id"] = batch1_airr["cell_id"].astype(str) + "_Batch1"
    batch2_airr = pandas.read_csv(os.path.join(work_folder, airr_file_2), sep="\t")
    batch2_airr["cell_id"] = batch2_airr["cell_id"].astype(str) + "_Batch2"

    adata_ir = ir.io.read_airr([pandas.concat([batch1_airr, batch2_airr])])
else:
    print("Processing single sample AIRR file before merging into MuData object.")
    adata_ir = ir.io.read_airr(os.path.join(work_folder, airr_file))

adata_ir.shape

# shape for BCR or TCR data should be (n, 0)

In [ ]:
# Check my adata_ir object columns

adata_ir.obs.head()

In [ ]:
adata_ir.obs.columns

## Preprocessing and BCR Quality Control ##

https://scirpy.scverse.org/en/latest/api.html

In [ ]:
# Print data information - mdata (transcriptomic)

mdata

In [ ]:
# Print data information - adata (AIRR)

adata_ir

In [ ]:
# Quality control. Can get a lot of this info out of Cellismo too. 
# No filters applied here because CLL has atypical and unproductive sequences that may be clonally relevant.
# Selects primary and secondary chains according to the IR model. 

ir.pp.index_chains(adata_ir)
ir.tl.chain_qc(adata_ir)


## Merge transcriptomic and BCR AIRR data

In [ ]:
# ir.pp.merge_with_ir(mdata, adata_ir) - this is depreciated
# Use MuData

mdata_airr = MuData({"mdata": mdata, "airr": adata_ir})

In [ ]:
# Check new mdata_airr object

mdata_airr.obs.head()

In [ ]:
# List modality keys of mdata_airr - this is not generating connectivity keys, whichis an issue downstream. 

list(mdata_airr.mod.keys())

## Confirm removal of CLL clones

This is a truncated version of BCR.ipynb used to confirm removal CLL clones by clone_id and junction_nt (CDR3 sequence plus 1 codon on either side). 

In [ ]:
# Pre-processing - this part must be run for ALL analyses. 
# Requires the same parameters for metric and sequence as the scirpy.tl.define_clonotype_clusters fucntion. Note, the defaults for these two functions are NOT the same!
# BCR recommended parameters are in place here from https://scirpy.scverse.org/en/latest/generated/scirpy.pp.ir_dist.html#scirpy.pp.ir_dist.  

ir.pp.ir_dist(mdata_airr, metric="normalized_hamming", sequence="nt", histogram=True, cutoff=15)


## Clonotypes

In [ ]:
# Define clonotypes

ir.tl.define_clonotypes (mdata_airr, key_added='clone_id') 

In [ ]:
# Clonotype clusters per sample or per group.
# Clone_id key adds clone_id and clone_id size to obs. This can be customised if it needs to be annotate per batch etc. 

ir.tl.define_clonotype_clusters(
    mdata_airr,
    sequence='nt',
    metric='normalized_hamming',
    receptor_arms='all',
    dual_ir='any',
    same_v_gene=True,
    same_j_gene=True,
    within_group='mdata:Sample_Name',
    partitions="fastgreedy",
    key_added="clone_id",
)

In [ ]:
# Clonotype network generation.

ir.tl.clonotype_network(
    mdata_airr, sequence='nt', metric='normalized_hamming', min_cells=3, clonotype_key="clone_id"
)

In [ ]:
mdata_airr

In [ ]:
mdata_airr.obs.columns

In [ ]:
# Plot clonotype network - visualisation of which samples are likely to have CLL clones, and their clone_id. 
# It still plots multiplets, undetermined and unassigned, but colours as 'nan'. 
# Subsampled data will be the only ones coloured in here, not very useful. 

_ = ir.pl.clonotype_network(
    mdata_airr,
    color="mdata:Sample_Name",
    title='Clonotype Network BCR by Sample Name',
    label_fontsize=9,
    base_size=20,
)

## Repertoire Analysis

In [ ]:
# Clonal Expansion - puts the clones into groups of 1, 2 and >2. 
# States number of clones that are >= 2
# Adjust breakpoints parameter

#ir.tl.clonal_expansion(mdata_airr, target_col='clone_id', breakpoints=(1, 2),)

ir.tl.clonal_expansion(
    mdata_airr, target_col='clone_id',
    breakpoints=(
        1, 
        2,
    )
)
expanded = np.sum(mdata_airr.obs["airr:clonal_expansion"] != "<= 1") / len(mdata_airr)
print(f"{expanded: .2%} of cells belong to expanded (size >= 2) clonotype cluster in this dataset.")

In [ ]:
# Plot group abundance - viualisation of samples that are likely to have CLL clones, but the number of cells with >2. 

#ir.pl.group_abundance(mdata_airr, target_col="has_ir",groupby="airr:clonalexpansion", max_cols=10)
ir.pl.group_abundance(
    mdata_airr, target_col="mdata:Sample_Name", groupby="airr:clonal_expansion", sort=["<= 1", "<= 2", "> 2"]
) 
#max_cols=10)

# some sample_names are missing because the control dataset has removed data for them, but left the sample name. Need to remake the h5mu.

#ir.pl.group_abundance(mdata_airr, groupby="mdata:Sample_Tag_Name")

In [ ]:
# Summarise clonal expansion by group - use when groups defined - see documetation https://scverse.org/scirpy/tags/v0.11.0/generated/scirpy.tl.summarize_clonal_expansion.html#scirpy.tl.summarize_clonal_expansion
# This shows which samples have n number of cells with >2 BCRs that are the same. This identifies which samples are likely to have CLL clones. 

ir.tl.summarize_clonal_expansion(mdata_airr, groupby="mdata:Sample_Name", breakpoints=(1,2,5,10,50))

## Clonotype Diversity

In [ ]:
# Alpha diversity of clonotypes within a group - set when groups defined # https://scverse.org/scirpy/tags/v0.11.0/generated/scirpy.tl.alpha_diversity.html#scirpy.tl.alpha_diversity
# Alpha diversity is within-sample and measures the "richness and eveness" of the repertoire. https://academic.oup.com/bioinformatics/article/40/7/btae431/7702327

import skbio as sk

ir.tl.alpha_diversity(mdata_airr, groupby='mdata:Sample_Name', target_col='clone_id', metric='normalized_shannon_entropy') 


In [ ]:
# Alpha diverity - needs groupby.
# Low diversity = more likely to have a monoclonal CLL clone. 

ir.pl.alpha_diversity(
    mdata_airr,
    groupby='mdata:Sample_Name',
)

## Clonotype Abundance ##

In [ ]:
# Group abundance
# Clone ID per sample.
# Shows, very well, which ones have high abundance. 

_= ir.pl.group_abundance(
    mdata_airr,
    groupby='clone_id',
    target_col='mdata:Sample_Name',
    max_cols=30,
    fig_kws={"dpi": 120}, figsize=[8, 8],
)

In [ ]:
mdata_airr.obs["airr:clone_id"]

# Recheck remain clone details if required. 

In [ ]:
# Return the nucleotide sequence of a specific clone_id - I can then find these in Cellismo (listed alphabetically) and delete the relevant clone.
# All clones have the same nucleotide sequence (this has been checked, but returns a large list for high frequence clones)
# Junction is used because this includes the CDR3 plus the two flanking conserved codons. https://docs.airr-community.org/en/v1.2.1/datarep/rearrangements.html, AIRR uses the IMGT definitions. 

clone_num ="8587"

list(mdata_airr.obs[mdata_airr.obs["airr:clone_id"]==clone_num]["mdata:BCR_Heavy_CDR3_Junction_Nucleotide_Dominant"].values[0:1])


In [ ]:
# Return the translation sequence of a specific clone_id - I can then find these in Cellismo, but nucleotide sequence preferred for deleting sequences due to >1nt used for some aa codons.
# I can check translation with the known VDJ sequence of the clone if this has been previously reported (HMDS).
# All clones have the same nucleotide sequence (this has been checked, but returns a large list for high frequence clones)
# Junction is used because this includes the CDR3 plus the two flanking conserved codons. https://docs.airr-community.org/en/v1.2.1/datarep/rearrangements.html, AIRR uses the IMGT definitions. 


list(
    mdata_airr.obs[mdata_airr.obs["airr:clone_id"]==clone_num]["mdata:BCR_Heavy_CDR3_Junction_Translation_Dominant"].values[
        0:1
    ]
)


In [ ]:
# Find the size of the clone_id - the largest one in the controls will be the cut-off for a 'normal' clone. 

#### CHANGE THE CLONE ID NUMBER ####

list(mdata_airr.obs[mdata_airr.obs["airr:clone_id"]==clone_num]["airr:clone_id_size"].values[0:1])

## Results ##

Make results notes here

## Definition of normal
1) Evaluate the normal control group of the dataset run here in BCR_identifyCLLclones.ipynb.
2) Using the notebook below, define the upper limit of normal clonal expansion as a % of total cells for each normal sample.
3) Note the range of normal clonal expansion as a %